# MNLI Benchmark Analysis — Groups 1 & 2

Fetches all MNLI runs from W&B across heterogeneity conditions,
computes derived metrics, prints per-group and cross-alpha summaries,
and saves to `results/mnli_all.pkl`.

**Temporal anchor — RTA (Round-to-Accuracy).** RTA is the first round where
evaluation accuracy reaches a single common target and stays at or above it
for `SUSTAIN` consecutive rounds (sustained crossing filters jitter spikes).
The target is shared by every heterogeneity condition and set to the highest
accuracy every run can reach (the weakest run's BestAcc, minus a small safety
margin). All system-cost metrics (MCD, StragOH) are averaged over rounds
1..RTA. `MCD = TTA / RTA`, so `TTA = RTA x MCD` is exact. BestAcc is kept over
the full budget as a quality safeguard. JFI is read at RTA; JFI@Rmax is kept
as a diagnostic only.

In [26]:
import os
import pickle
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import wandb

warnings.filterwarnings("ignore")

## Configuration

In [27]:
STRATEGIES = ["random", "fedcs", "tifl", "oort"]
ALPHAS = [0.5, 2.0, "iid"]  # ordered: high non-IID -> moderate -> IID
ALPHA_LABELS = {0.5: "α=0.5", 2.0: "α=2", "iid": "IID"}

# ── RTA temporal anchor ─────────────────────────────────────────
# A single common target accuracy shared across all conditions.
#   "auto": derive from data as (min BestAcc over all runs) - TARGET_SAFETY,
#           floored to TARGET_STEP. Guarantees every run reaches it.
#   or set a fixed float (e.g. 0.78) to override.
TARGET_MODE   = "auto"
TARGET_SAFETY = 0.01    # set the target this far below the weakest run's BestAcc
TARGET_STEP   = 0.01    # round the auto target down to this granularity
SUSTAIN       = 3       # accuracy must stay >= target for this many rounds

# Experiment constants
R_MAX = 100
N_CLIENTS = 300
K = 30  # clients selected per round (for the coverage-floor diagnostic)

OUTPUT_PATH = "results/mnli_all.pkl"
TARGET_PATH = "results/mnli_target.json"
os.makedirs("results", exist_ok=True)


In [28]:
ENTITY  = "ilham-abdillah-alhamdi-universitas-indonesia"
PROJECT = "u-flora-experiments"

# ── Group names per alpha condition ──────────────────────────────────────
GROUPS = {
    2.0:   "final-group1-mnli",
    0.5:   "final-group2-mnli-a-0.5",
    "iid": "final-group2-mnli-iid",
}

def _parse_alpha_key(partition: str, raw_alpha) -> float | str:
    """Determine the alpha key for RUNS dict from partition strategy and raw alpha."""
    if partition == "iid":
        return "iid"
    # Dirichlet: normalize to canonical float key
    val = float(raw_alpha)
    for canonical in [0.5, 2.0]:
        if abs(val - canonical) < 0.01:
            return canonical
    return val


In [29]:
import wandb
api = wandb.Api(timeout=60)

RUNS = {}

print("Fetching run IDs from W&B groups...\n")

for alpha_key, group_name in GROUPS.items():
    print(f"  Group '{group_name}' (alpha={alpha_key}):")
    group_runs = api.runs(f"{ENTITY}/{PROJECT}", filters={"group": group_name})

    found = []
    for run in group_runs:
        try:
            partition  = run.config["dataset"]["partition"]["strategy"]
            raw_alpha  = run.config["dataset"]["partition"]["dirichlet"]["alpha"]
            strategy   = run.config["strategy"]["name"].lower().strip()
            seed       = int(run.config["seed"])
            parsed_key = _parse_alpha_key(partition, raw_alpha)
        except (KeyError, TypeError, ValueError) as e:
            print(f"    ✗ Run {run.id} ({run.name}): could not parse config — {e}")
            continue

        # Only keep runs that belong to the expected alpha_key for this group
        if parsed_key != alpha_key:
            print(f"    ⚠  Skipping {run.id}: parsed alpha={parsed_key} != expected {alpha_key}")
            continue

        if alpha_key not in RUNS:
            RUNS[alpha_key] = {}
        if strategy not in RUNS[alpha_key]: 
            RUNS[alpha_key][strategy] = {}

        if seed in RUNS[alpha_key].get(strategy, {}):
            print(f"    ⚠  Duplicate: {strategy} seed={seed} — "
                  f"keeping {RUNS[alpha_key][strategy][seed]}, ignoring {run.id}")
        else:
            RUNS[alpha_key][strategy][seed] = run.id
            found.append((strategy, seed, run.id, run.name))

    for strategy, seed, run_id, run_name in sorted(found):
        print(f"    ✓ {strategy:8s}  seed={seed:3d}  {run_id}  ({run_name})")

    # Validate expected seeds are present
    expected_seeds = {2.0: [42, 123, 456], 0.5: [42, 123, 456], "iid": [42]}
    for strat in STRATEGIES:
        expected = expected_seeds.get(alpha_key, [42])
        present  = list(RUNS.get(alpha_key, {}).get(strat, {}).keys())
        missing  = [s for s in expected if s not in present]
        if missing:
            print(f"    ✗ MISSING: {strat} seeds {missing}")
    print()

print("Done. RUNS structure:")
for alpha in RUNS:
    for strat in RUNS.get(alpha, {}):
        seeds = list(RUNS[alpha][strat].keys())
        print(f"  alpha={alpha}  {strat:8s}  seeds={seeds}")

Fetching run IDs from W&B groups...

  Group 'final-group1-mnli' (alpha=2.0):
    ✓ fedcs     seed= 42  r39tsb13  (fedcs-mnli-42-a2-v1-20260506_09:35:22)
    ✓ fedcs     seed=123  qqarz0um  (fedcs-mnli-123-a2-v1-20260508_13:44:56)
    ✓ fedcs     seed=456  3z3lad4v  (fedcs-mnli-456-a2-v1-20260508_22:57:59)
    ✓ oort      seed= 42  7ij1j97y  (oort-mnli-42-a2-v5-20260604_19:40:26)
    ✓ oort      seed=123  8712izpi  (oort-mnli-123-a2-v5-20260604_19:00:55)
    ✓ oort      seed=456  d4woht37  (oort-mnli-456-a2-v5-20260604_18:22:38)
    ✓ random    seed= 42  ukqx8xyv  (random-mnli-42-a2-v1-20260506_05:21:22)
    ✓ random    seed=123  jhx7m47m  (random-mnli-123-a2-v1-20260508_11:23:50)
    ✓ random    seed=456  teapg762  (random-mnli-456-a2-v1-20260508_20:37:12)
    ✓ tifl      seed= 42  9jtvaulj  (tifl-mnli-42-a2-v2-20260507_20:52:12)
    ✓ tifl      seed=123  i6znx4zl  (tifl-mnli-123-a2-v1-20260508_15:44:19)
    ✓ tifl      seed=456  8m686cay  (tifl-mnli-456-a2-v1-20260509_00:56:00)

  Gr

## W&B Columns & Utility Functions

In [30]:
HISTORY_KEYS = [
    # ── Round-level ────────────────────────────────────────────────────
    "round/server_round",
    "round/wall_clock",
    "round/cumulative_wall_clock",
    "round/duration_mean",
    "round/duration_std",
    "round/num_client_selected",
    "round/num_client_completed",
    # ── Evaluation ────────────────────────────────────────────────────
    "eval/accuracy",
    "eval/loss",
    # ── Fairness ──────────────────────────────────────────────────────
    "fairness/jain_index",
    "fairness/unique_clients_explored",
    "fairness/exploration_ratio",
    # ── Utility signal (for RQ2 mechanism analysis) ───────────────────
    "utility/loss_cv",
    "utility/loss_std",
    "utility/loss_rms_cv",
    "utility/loss_rms_std",
    # ── Oort-specific (will be NaN for non-Oort runs) ─────────────────
    "strategy/oort_preferred_t",
    "strategy/oort_epsilon",
    "strategy/oort_round_threshold",
]

In [31]:
def sustained_crossing(series: pd.Series, target: float, sustain: int = 1) -> int | None:
    """1-indexed round of the first value >= target that then stays >= target
    for `sustain` consecutive rounds (or until the run ends).

    Sustained crossing filters single-round jitter spikes, which is the whole
    point of moving the temporal anchor off the patience-based R*.
    """
    vals = series.ffill().values.astype(float)
    n = len(vals)
    for r in range(n):
        if vals[r] >= target:
            end = min(r + sustain, n)
            if np.all(vals[r:end] >= target):
                return r + 1
    return None


def safe_iloc(series: pd.Series, idx_1based):
    """0-indexed access with bounds check. Returns None on miss/NaN/None index."""
    if idx_1based is None:
        return None
    i = idx_1based - 1
    if i < 0 or i >= len(series):
        return None
    v = series.iloc[i]
    return None if pd.isna(v) else float(v)


def safe_col(hist: pd.DataFrame, key: str) -> pd.Series:
    """Return column if it exists, else NaN series of same length."""
    if key in hist.columns:
        return hist[key]
    return pd.Series([np.nan] * len(hist), index=hist.index)


## Fetch & Compute Pipeline

In [32]:
def fetch_run(run_id: str) -> tuple[pd.DataFrame, dict]:
    api = wandb.Api(timeout=60)
    run = api.run(f"{ENTITY}/{PROJECT}/{run_id}")

    hist = run.history(samples=500, pandas=True)

    if "round/server_round" not in hist.columns:
        raise ValueError(f"Run {run_id}: 'round/server_round' column missing")

    hist = hist.dropna(subset=["round/server_round"]).copy()
    hist["round/server_round"] = hist["round/server_round"].astype(int)
    hist = hist.sort_values("round/server_round").reset_index(drop=True)

    summary = dict(run.summary)
    return hist, summary

In [33]:
def compute_base_metrics(hist: pd.DataFrame, summary: dict, run_id: str) -> dict:
    """Target-independent metrics + raw per-round series. The RTA-anchored
    metrics are filled later by compute_anchored_metrics once the common
    target has been derived from every run's BestAcc."""

    rounds = hist["round/server_round"]
    acc = hist["eval/accuracy"].ffill()
    loss = safe_col(hist, "eval/loss")
    cum_wc = hist["round/cumulative_wall_clock"]
    wall_clock = hist["round/wall_clock"]
    dur_mean = hist["round/duration_mean"]
    dur_std = hist["round/duration_std"]
    jfi = hist["fairness/jain_index"]
    unique_exp = safe_col(hist, "fairness/unique_clients_explored")
    loss_cv = safe_col(hist, "utility/loss_cv")
    loss_rms_cv = safe_col(hist, "utility/loss_rms_cv")

    # Oort-specific (NaN for non-Oort)
    oort_preferred_t = safe_col(hist, "strategy/oort_preferred_t")
    oort_epsilon = safe_col(hist, "strategy/oort_epsilon")
    oort_round_threshold = safe_col(hist, "strategy/oort_round_threshold")

    # straggler ratio is target-independent; the overhead is windowed at RTA later
    strag_ratio = wall_clock / dur_mean.replace(0.0, np.nan)

    # ── Quality safeguard: BestAcc over the full budget ────────────
    best_acc = float(summary.get("summary/best_metric", acc.max()))
    acc_at_r_max = safe_iloc(acc, R_MAX) or float(acc.iloc[-1])

    return {
        "run_id": run_id,
        # ── Quality safeguard ───────────────────────────
        "best_acc": best_acc,
        "acc_at_r_max": acc_at_r_max,
        # ── RTA-anchored (filled by compute_anchored_metrics) ────────
        "rta": None, "tta": None, "mcd": None,
        "strag_overhead": None, "jfi_at_rta": None, "jfi_at_r_max": None,
        "unique_at_rta": None, "never_selected": None,
        # ── Series (for plotting + anchoring) ─────────────────
        "rounds": rounds,
        "acc_series": acc,
        "loss_series": loss,
        "cum_wc_series": cum_wc,
        "wall_clock_series": wall_clock,
        "dur_mean_series": dur_mean,
        "dur_std_series": dur_std,
        "strag_ratio_series": strag_ratio,
        "jfi_series": jfi,
        "unique_exp_series": unique_exp,
        "loss_cv_series": loss_cv,
        "loss_rms_cv_series": loss_rms_cv,
        "oort_preferred_t_series": oort_preferred_t,
        "oort_epsilon_series": oort_epsilon,
        "oort_round_threshold_series": oort_round_threshold,
    }


def compute_anchored_metrics(m: dict, target: float) -> dict:
    """Fill RTA and the rounds-1..RTA system-cost metrics for one run.

    MCD (Mean Cohort Duration) is the average per-client duration in the
    selected cohort over rounds 1..RTA. It measures the level of cohort speed
    and is independent of StragOH, which measures the within-round spread. The
    wall-clock decomposition is approximate, TTA ~= RTA * MCD * (1 + StragOH).
    """
    acc = pd.Series(m["acc_series"]).reset_index(drop=True).ffill()
    cum_wc = pd.Series(m["cum_wc_series"]).reset_index(drop=True)
    jfi = pd.Series(m["jfi_series"]).reset_index(drop=True)
    unique_exp = pd.Series(m["unique_exp_series"]).reset_index(drop=True)
    strag_ratio = pd.Series(m["strag_ratio_series"]).reset_index(drop=True)

    rta = sustained_crossing(acc, target, SUSTAIN)
    m["rta"] = rta

    # JFI@Rmax: kept as a diagnostic (full-coverage equity check), not headlined.
    m["jfi_at_r_max"] = safe_iloc(jfi, R_MAX) or float(jfi.iloc[-1])

    if rta is None:
        return m  # target never sustained-reached; anchored cost metrics stay None

    tta = safe_iloc(cum_wc, rta)
    m["tta"] = tta
    # m["mcd"] = (tta / rta) if (tta is not None and rta) else None # MRD
    dur_mean = pd.Series(m["dur_mean_series"]).reset_index(drop=True)
    m["mcd"] = float(dur_mean.iloc[:rta].mean()) if rta else None
    m["strag_overhead"] = float((strag_ratio.iloc[:rta] - 1.0).clip(lower=0).mean())
    m["jfi_at_rta"] = safe_iloc(jfi, rta)
    u = safe_iloc(unique_exp, rta)
    m["unique_at_rta"] = u
    m["never_selected"] = int(N_CLIENTS - u) if u is not None else None
    return m


## Main: Fetch All Runs

In [34]:
all_results = {}  # {alpha: {strategy: {seed: metrics_dict}}}

for alpha in ALPHAS:
    print(f"\n{'='*70}")
    print(f"  Alpha = {ALPHA_LABELS[alpha]}")
    print(f"{'='*70}")
    all_results[alpha] = {}

    for strategy in STRATEGIES:
        all_results[alpha][strategy] = {}
        seeds = list(RUNS[alpha][strategy].keys())

        for seed in seeds:
            run_id = RUNS[alpha][strategy][seed]
            print(f"  {strategy:6s} seed={seed:3d}  ({run_id}) ...", end=" ", flush=True)
            try:
                hist, summary = fetch_run(run_id)
                metrics = compute_base_metrics(hist, summary, run_id)
                all_results[alpha][strategy][seed] = metrics
                print(f"OK  best={metrics['best_acc']:.4f}")
            except Exception as e:
                print(f"FAILED: {e}")
                all_results[alpha][strategy][seed] = None



  Alpha = α=0.5
  random seed= 42  (kwo6g2hd) ... OK  best=0.8240
  random seed=123  (7y9d0k79) ... OK  best=0.8424
  random seed=456  (t2cilmtb) ... OK  best=0.8370
  fedcs  seed= 42  (44p9iwlr) ... OK  best=0.7898
  fedcs  seed=123  (qkdc4s9t) ... OK  best=0.7953
  fedcs  seed=456  (ozw36g2z) ... OK  best=0.7902
  tifl   seed= 42  (3ius4tsd) ... OK  best=0.8234
  tifl   seed=123  (h3do7iw7) ... OK  best=0.8336
  tifl   seed=456  (rrvlijkj) ... OK  best=0.8330
  oort   seed=456  (p9vh3grs) ... OK  best=0.8162
  oort   seed=123  (pkkz91w4) ... OK  best=0.8210
  oort   seed= 42  (ekm0pwpb) ... OK  best=0.8208

  Alpha = α=2
  random seed= 42  (ukqx8xyv) ... OK  best=0.8537
  random seed=123  (jhx7m47m) ... OK  best=0.8567
  random seed=456  (teapg762) ... OK  best=0.8498
  fedcs  seed= 42  (r39tsb13) ... OK  best=0.8260
  fedcs  seed=123  (qqarz0um) ... OK  best=0.8326
  fedcs  seed=456  (3z3lad4v) ... OK  best=0.8370
  tifl   seed= 42  (9jtvaulj) ... OK  best=0.8474
  tifl   seed=123 

## Derive Common Target & Anchor Runs

In [35]:
# ── Derive the common target from BestAcc, then anchor every run ────────
def _all_runs():
    for alpha in ALPHAS:
        for strat in STRATEGIES:
            for seed, m in all_results[alpha][strat].items():
                if m is not None:
                    yield alpha, strat, seed, m

# BestAcc matrix (mean across seeds) so the binding cell is visible
print("BestAcc per condition × strategy (mean across seeds):\n")
hdr = f"{'Strategy':<8}" + "".join(f"{ALPHA_LABELS[a]:>10}" for a in ALPHAS)
print(hdr); print("-" * len(hdr))
for strat in STRATEGIES:
    row = f"{strat:<8}"
    for alpha in ALPHAS:
        vals = [m["best_acc"] for a, s, sd, m in _all_runs() if a == alpha and s == strat]
        row += f"{(np.mean(vals) if vals else float('nan')):>10.4f}"
    print(row)

best_accs = [m["best_acc"] for *_, m in _all_runs()]
min_best  = min(best_accs)
weakest   = min(
    ((a, s, sd, m["best_acc"]) for a, s, sd, m in _all_runs()),
    key=lambda x: x[3],
)

if TARGET_MODE == "auto":
    COMMON_TARGET = np.floor((min_best - TARGET_SAFETY) / TARGET_STEP) * TARGET_STEP
    COMMON_TARGET = round(float(COMMON_TARGET), 4)
else:
    COMMON_TARGET = float(TARGET_MODE)

print(f"\nWeakest run BestAcc = {min_best:.4f}  "
      f"({ALPHA_LABELS[weakest[0]]} / {weakest[1]} / seed={weakest[2]})")
print(f"Common target accuracy = {COMMON_TARGET:.4f}  "
      f"(mode={TARGET_MODE}, sustain={SUSTAIN})")

# Anchor every run to the common target
n_undef = 0
for alpha, strat, seed, m in _all_runs():
    compute_anchored_metrics(m, COMMON_TARGET)
    if m["rta"] is None:
        n_undef += 1
        print(f"  ⚠ RTA undefined: {ALPHA_LABELS[alpha]} / {strat} / seed={seed}")
print(f"\nAnchored all runs. RTA undefined in {n_undef} run(s).")


BestAcc per condition × strategy (mean across seeds):

Strategy     α=0.5       α=2       IID
--------------------------------------
random      0.8345    0.8534    0.8470
fedcs       0.7918    0.8319    0.8457
tifl        0.8300    0.8453    0.8525
oort        0.8193    0.8440    0.8490

Weakest run BestAcc = 0.7898  (α=0.5 / fedcs / seed=42)
Common target accuracy = 0.7700  (mode=auto, sustain=3)

Anchored all runs. RTA undefined in 0 run(s).


## Summary Helpers

In [36]:
def get_seeds(alpha, strategy):
    """Return list of seeds that have valid data for this (alpha, strategy)."""
    return [
        s for s in all_results[alpha][strategy]
        if all_results[alpha][strategy][s] is not None
    ]


def agg(alpha, strategy, key):
    """Mean ± std of a scalar metric across seeds. Returns (mean, std, n)."""
    vals = [
        all_results[alpha][strategy][s][key]
        for s in get_seeds(alpha, strategy)
        if all_results[alpha][strategy][s].get(key) is not None
    ]
    if not vals:
        return None, None, 0
    return float(np.mean(vals)), float(np.std(vals)), len(vals)


def fmt(mean, std, n, decimals=2):
    """Format mean±std, or just mean if n=1."""
    if mean is None:
        return "—"
    if n <= 1:
        return f"{mean:.{decimals}f}"
    return f"{mean:.{decimals}f}±{std:.{decimals}f}"

## Group 1 Summary (α=2)

In [37]:
def print_group_summary(alpha):
    label = ALPHA_LABELS[alpha]
    seeds_for_alpha = list(RUNS[alpha][STRATEGIES[0]].keys())

    # ── Per-seed table ───────────────────────────────
    header = (
        f"{'Strategy':<8} {'Seed':>4}  {'RTA':>4}  {'TTA(h)':>7}  {'MCD(s)':>7}  "
        f"{'BestAcc':>7}  {'JFI@RTA':>8}  {'Uniq':>4}  {'Excl':>4}  {'StragOH':>7}"
    )
    sep = "=" * len(header)
    print(f"\n{'─'*70}\n  Group Summary: {label}\n{'─'*70}")
    print(f"\n{sep}\n{header}\n{sep}")

    for strategy in STRATEGIES:
        for seed in seeds_for_alpha:
            if seed not in all_results[alpha][strategy]:
                continue
            m = all_results[alpha][strategy][seed]
            if m is None:
                print(f"{strategy:<8} {seed:>4}  MISSING / SKIPPED")
                continue
            rta_s  = str(m["rta"]) if m["rta"] else "—"
            tta_s  = f"{m['tta']/3600:.2f}" if m["tta"] else "—"
            MCD_s  = f"{m['mcd']:.1f}" if m["mcd"] is not None else "—"
            jfi_s  = f"{m['jfi_at_rta']:.3f}" if m["jfi_at_rta"] is not None else "—"
            uniq_s = f"{m['unique_at_rta']:.0f}" if m["unique_at_rta"] is not None else "—"
            excl_s = f"{m['never_selected']}" if m["never_selected"] is not None else "—"
            soh_s  = f"{m['strag_overhead']:.3f}" if m["strag_overhead"] is not None else "—"
            print(
                f"{strategy:<8} {seed:>4}  {rta_s:>4}  {tta_s:>7}  {MCD_s:>7}  "
                f"{m['best_acc']:>7.4f}  {jfi_s:>8}  {uniq_s:>4}  {excl_s:>4}  {soh_s:>7}"
            )
        print("-" * len(header))

    # ── Seed-aggregated table ─────────────────────────
    print(f"\n── Seed-aggregated (mean ± std) ──")
    agg_header = (
        f"{'Strategy':<8}  {'RTA':>10}  {'TTA(h)':>10}  {'MCD(s)':>10}  "
        f"{'BestAcc':>10}  {'JFI@RTA':>10}  {'NeverSel':>10}  {'StragOH':>10}"
    )
    print(agg_header)
    print("-" * len(agg_header))

    for strategy in STRATEGIES:
        n = len(get_seeds(alpha, strategy))
        rta_m, rta_sd, _ = agg(alpha, strategy, "rta")
        tta_vals = [
            all_results[alpha][strategy][s]["tta"] / 3600
            for s in get_seeds(alpha, strategy)
            if all_results[alpha][strategy][s].get("tta") is not None
        ]
        tta_m  = np.mean(tta_vals) if tta_vals else None
        tta_sd = np.std(tta_vals) if tta_vals else None
        MCD_m, MCD_sd, _ = agg(alpha, strategy, "mcd")
        ba_m,  ba_sd,  _ = agg(alpha, strategy, "best_acc")
        j_m,   j_sd,   _ = agg(alpha, strategy, "jfi_at_rta")
        ns_m,  ns_sd,  _ = agg(alpha, strategy, "never_selected")
        soh_m, soh_sd, _ = agg(alpha, strategy, "strag_overhead")

        print(
            f"{strategy:<8}  {fmt(rta_m,rta_sd,n,1):>10}  {fmt(tta_m,tta_sd,n,2):>10}  "
            f"{fmt(MCD_m,MCD_sd,n,1):>10}  {fmt(ba_m,ba_sd,n,4):>10}  "
            f"{fmt(j_m,j_sd,n,3):>10}  {fmt(ns_m,ns_sd,n,0):>10}  {fmt(soh_m,soh_sd,n,3):>10}"
        )


# Print Group 1
print_group_summary(2.0)



──────────────────────────────────────────────────────────────────────
  Group Summary: α=2
──────────────────────────────────────────────────────────────────────

Strategy Seed   RTA   TTA(h)   MCD(s)  BestAcc   JFI@RTA  Uniq  Excl  StragOH
random     42    19     7.01    472.3   0.8537     0.659   256    44    1.968
random    123    22     7.97    493.3   0.8567     0.714   274    26    1.769
random    456    23     7.81    485.3   0.8498     0.682   263    37    1.625
-----------------------------------------------------------------------------
fedcs      42    41     3.65    203.3   0.8260     0.214    79   221    0.618
fedcs     123    28     2.38    192.9   0.8326     0.210    75   225    0.645
fedcs     456    24     2.01    196.0   0.8370     0.204    73   227    0.603
-----------------------------------------------------------------------------
tifl       42    24     3.78    414.4   0.8474     0.667   279    21    0.379
tifl      123    24     6.69    608.0   0.8450     0.59

## Group 2: Cross-Alpha Comparison

In [38]:
def print_cross_alpha():
    """Compare key metrics across heterogeneity levels for each strategy."""

    metrics_to_compare = [
        ("RTA", "rta", 1),
        ("TTA (h)", None, 2),  # special handling: convert to hours
        ("Best Acc", "best_acc", 4),
        ("JFI@RTA", "jfi_at_rta", 3),
        ("MCD (s)", "mcd", 1),
        ("Strag OH", "strag_overhead", 3),
        ("Never Sel", "never_selected", 0),
    ]

    print(f"\n{'='*70}")
    print(f"  GROUP 2: Cross-Alpha Comparison")
    print(f"{'='*70}")

    for metric_label, metric_key, decimals in metrics_to_compare:
        alpha_headers = [f"{ALPHA_LABELS[a]:>14}" for a in ALPHAS]
        print(f"\n── {metric_label} ──")
        print(f"{'Strategy':<8}  " + "  ".join(alpha_headers))
        print("-" * (10 + 16 * len(ALPHAS)))

        for strategy in STRATEGIES:
            row = [f"{strategy:<8}"]
            for alpha in ALPHAS:
                n = len(get_seeds(alpha, strategy))
                if n == 0:
                    row.append(f"{'—':>14}")
                    continue

                if metric_key is None and metric_label == "TTA (h)":
                    vals = [
                        all_results[alpha][strategy][s]["tta"] / 3600
                        for s in get_seeds(alpha, strategy)
                        if all_results[alpha][strategy][s] is not None
                        and all_results[alpha][strategy][s].get("tta") is not None
                    ]
                    if not vals:
                        row.append(f"{'—':>14}")
                    elif len(vals) == 1:
                        row.append(f"{vals[0]:>14.{decimals}f}")
                    else:
                        row.append(f"{np.mean(vals):.{decimals}f}±{np.std(vals):.{decimals}f}".rjust(14))
                else:
                    m, s, ct = agg(alpha, strategy, metric_key)
                    row.append(f"{fmt(m, s, ct, decimals):>14}")

            print("  ".join(row))


print_cross_alpha()



  GROUP 2: Cross-Alpha Comparison

── RTA ──
Strategy           α=0.5             α=2             IID
----------------------------------------------------------
random          42.3±6.1        21.3±1.7            22.0
fedcs          77.3±15.6        31.0±7.3            20.0
tifl            40.0±8.5        25.0±1.4            18.0
oort           52.3±16.0        27.7±2.9            20.0

── TTA (h) ──
Strategy           α=0.5             α=2             IID
----------------------------------------------------------
random        22.95±3.90       7.59±0.42            7.30
fedcs          5.61±1.75       2.68±0.70            1.92
tifl           7.58±1.08       5.03±1.22            2.90
oort          11.76±2.58       6.57±0.65            5.01

── Best Acc ──
Strategy           α=0.5             α=2             IID
----------------------------------------------------------
random     0.8345±0.0077   0.8534±0.0028          0.8470
fedcs      0.7918±0.0025   0.8319±0.0045          0.8457
tifl 

In [39]:
for alpha in ALPHAS:
    if alpha == 2.0:
        continue  # already printed above
    print_group_summary(alpha)


──────────────────────────────────────────────────────────────────────
  Group Summary: α=0.5
──────────────────────────────────────────────────────────────────────

Strategy Seed   RTA   TTA(h)   MCD(s)  BestAcc   JFI@RTA  Uniq  Excl  StragOH
random     42    42    24.18    482.1   0.8240     0.812   297     3    3.405
random    123    35    17.67    491.7   0.8424     0.779   293     7    2.752
random    456    50    26.98    495.8   0.8370     0.844   297     3    2.844
-----------------------------------------------------------------------------
fedcs      42    70     4.64    126.6   0.7898     0.217    82   218    0.915
fedcs     123    63     4.12    121.6   0.7953     0.216    78   222    0.970
fedcs     456    99     8.07    192.9   0.7902     0.229    90   210    0.615
-----------------------------------------------------------------------------
tifl       42    47     8.96    432.4   0.8234     0.631   248    52    0.548
tifl      123    28     6.31    524.1   0.8336     0.

## RTA Coverage-Floor Diagnostic

In [40]:
# ── RTA coverage-floor diagnostic: is JFI@RTA meaningful? ──────────────
# With K=30 picks / N=300 clients, no strategy can touch every client before
# round N/K = 10. Where RTA is below ~10-15, JFI@RTA partly measures coverage
# incompleteness rather than selection bias. This table exposes that so we can
# decide whether JFI@RTA alone is enough, or whether to keep JFI@Rmax for the
# Oort full-coverage equity point.
min_cov_round = N_CLIENTS / K
print(f"Coverage floor: >= {min_cov_round:.0f} rounds needed to touch all {N_CLIENTS} clients once.\n")

def _s(v, d):
    return "—" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.{d}f}"

hdr = f"{'Cond':>7} {'Strategy':<8} {'RTA':>5} {'cov@RTA%':>9} {'JFI@RTA':>8} {'JFI@Rmax':>9}"
print(hdr); print("-" * len(hdr))
for alpha in ALPHAS:
    for strat in STRATEGIES:
        rta_m = agg(alpha, strat, "rta")[0]
        jrt_m = agg(alpha, strat, "jfi_at_rta")[0]
        jmx_m = agg(alpha, strat, "jfi_at_r_max")[0]
        uq_m  = agg(alpha, strat, "unique_at_rta")[0]
        cov   = (uq_m / N_CLIENTS * 100) if uq_m is not None else None
        print(f"{ALPHA_LABELS[alpha]:>7} {strat:<8} {_s(rta_m,0):>5} {_s(cov,1):>9} "
              f"{_s(jrt_m,3):>8} {_s(jmx_m,3):>9}")
    print("-" * len(hdr))
print("\nRead: high cov@RTA -> JFI@RTA reflects genuine selection bias.")
print("      low cov@RTA (early RTA) -> JFI@RTA is partly a coverage floor;")
print("      use JFI@Rmax for the full-coverage equity comparison.")


Coverage floor: >= 10 rounds needed to touch all 300 clients once.

   Cond Strategy   RTA  cov@RTA%  JFI@RTA  JFI@Rmax
---------------------------------------------------
  α=0.5 random      42      98.6    0.811     0.912
  α=0.5 fedcs       77      27.8    0.221     0.221
  α=0.5 tifl        40      82.6    0.602     0.954
  α=0.5 oort        52     100.0    0.561     0.651
---------------------------------------------------
    α=2 random      21      88.1    0.685     0.912
    α=2 fedcs       31      25.2    0.209     0.214
    α=2 tifl        25      84.9    0.628     0.954
    α=2 oort        28     100.0    0.605     0.715
---------------------------------------------------
    IID random      22      89.3    0.688     0.916
    IID fedcs       20      26.0    0.210     0.215
    IID tifl        18      81.0    0.607     0.958
    IID oort        20     100.0    0.677     0.829
---------------------------------------------------

Read: high cov@RTA -> JFI@RTA reflects genuine 

## Save

In [41]:
import json

with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(all_results, f)

with open(TARGET_PATH, "w") as f:
    json.dump({"common_target": float(COMMON_TARGET), "sustain": int(SUSTAIN)}, f)

print(f"\nSaved → {OUTPUT_PATH}")
print(f"Saved → {TARGET_PATH}  (common_target={COMMON_TARGET:.4f}, sustain={SUSTAIN})")
print(f"Structure: all_results[alpha][strategy][seed] → dict of scalars + series")
for alpha in ALPHAS:
    for strat in STRATEGIES:
        n = len(get_seeds(alpha, strat))
        print(f"  {ALPHA_LABELS[alpha]:>6} / {strat:<8}: {n} seeds")



Saved → results/mnli_all.pkl
Saved → results/mnli_target.json  (common_target=0.7700, sustain=3)
Structure: all_results[alpha][strategy][seed] → dict of scalars + series
   α=0.5 / random  : 3 seeds
   α=0.5 / fedcs   : 3 seeds
   α=0.5 / tifl    : 3 seeds
   α=0.5 / oort    : 3 seeds
     α=2 / random  : 3 seeds
     α=2 / fedcs   : 3 seeds
     α=2 / tifl    : 3 seeds
     α=2 / oort    : 3 seeds
     IID / random  : 1 seeds
     IID / fedcs   : 1 seeds
     IID / tifl    : 1 seeds
     IID / oort    : 1 seeds


============== Reconstruct from Raw Table ==================

In [42]:
import json, glob
import pandas as pd

TABLE_COLUMNS = ["round", "node_id", "train_loss", "duration",
                 "compute_time", "communication_time", "num_samples", "selected_by"]

def fetch_history_table(run_id: str) -> pd.DataFrame | None:
    """Reconstruct the incremental client/raw_training_history table.

    Stored as a versioned run artifact (log_mode='INCREMENTAL'). We read every
    .table.json shard across all matching artifacts, concatenate, and
    de-duplicate on (round, node_id). 'node_id' holds the partition/client id.
    """
    run = api.run(f"{ENTITY}/{PROJECT}/{run_id}")
    frames = []
    for art in run.logged_artifacts():
        if "raw_training" not in art.name:
            continue
        try:
            art_dir = art.download()
        except Exception:
            continue
        for shard in glob.glob(f"{art_dir}/**/*.table.json", recursive=True):
            with open(shard) as f:
                tj = json.load(f)
            frames.append(pd.DataFrame(tj["data"], columns=tj.get("columns", TABLE_COLUMNS)))
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True)
    df = df.drop_duplicates(subset=["round", "node_id"]).reset_index(drop=True)
    df["round"] = df["round"].astype(int)
    df["node_id"] = df["node_id"].astype(int)
    return df


def selection_counts(table_df: pd.DataFrame, r_cutoff: int, n_clients: int = N_CLIENTS) -> np.ndarray:
    """Per-client selection count up to round r_cutoff.
    Returns length-n_clients array; clients never selected get 0.
    Each table row is one client-round participation."""
    sub = table_df[table_df["round"] <= r_cutoff]
    counts = sub.groupby("node_id").size()
    full = np.zeros(n_clients, dtype=int)
    for pid, c in counts.items():
        if 0 <= pid < n_clients:
            full[pid] = int(c)
    return full

In [19]:
print("Fetching raw training-history tables...\n")
for alpha in ALPHAS:
    for strategy in STRATEGIES:
        for seed in get_seeds(alpha, strategy):
            m = all_results[alpha][strategy][seed]
            run_id = m["run_id"]
            print(
                f"  {ALPHA_LABELS[alpha]:>6} {strategy:6s} seed={seed:3d} ({run_id}) ...",
                end=" ", flush=True,
            )
            try:
                tbl = fetch_history_table(run_id)
                if tbl is None:
                    print("NO TABLE")
                    m["sel_counts_rta"] = m["sel_counts_r_max"] = None
                    continue
                m["sel_counts_rta"] = selection_counts(tbl, m["rta"]) if m["rta"] else None
                m["sel_counts_r_max"] = selection_counts(tbl, R_MAX)
                m["history_table"] = tbl  # keep for any later table-based metric
                uniq = int((m["sel_counts_rta"] > 0).sum()) if m["sel_counts_rta"] is not None else 0
                print(f"OK  rows={len(tbl)}  unique@RTA={uniq}")
            except Exception as e:
                print(f"FAILED: {e}")
                m["sel_counts_rta"] = m["sel_counts_r_max"] = None


Fetching raw training-history tables...

   α=0.5 random seed= 42 (kwo6g2hd) ... 

wandb:   1 of 1 files downloaded.  
wandb:   2 of 2 files downloaded.  
wandb:   3 of 3 files downloaded.  
wandb:   4 of 4 files downloaded.  
wandb:   5 of 5 files downloaded.  
wandb:   6 of 6 files downloaded.  
wandb:   7 of 7 files downloaded.  
wandb:   8 of 8 files downloaded.  
wandb:   9 of 9 files downloaded.  
wandb:   10 of 10 files downloaded.  
wandb:   11 of 11 files downloaded.  
wandb:   12 of 12 files downloaded.  
wandb:   13 of 13 files downloaded.  
wandb:   14 of 14 files downloaded.  
wandb:   15 of 15 files downloaded.  
wandb:   16 of 16 files downloaded.  
wandb:   17 of 17 files downloaded.  
wandb:   18 of 18 files downloaded.  
wandb:   19 of 19 files downloaded.  
wandb:   20 of 20 files downloaded.  


OK  rows=2997  unique@RTA=297
   α=0.5 random seed=123 (7y9d0k79) ... 

wandb:   1 of 1 files downloaded.  
wandb:   2 of 2 files downloaded.  
wandb:   3 of 3 files downloaded.  
wandb:   4 of 4 files downloaded.  
wandb:   5 of 5 files downloaded.  
wandb:   6 of 6 files downloaded.  
wandb:   7 of 7 files downloaded.  
wandb:   8 of 8 files downloaded.  
wandb:   9 of 9 files downloaded.  
wandb:   10 of 10 files downloaded.  
wandb:   11 of 11 files downloaded.  
wandb:   12 of 12 files downloaded.  
wandb:   13 of 13 files downloaded.  
wandb:   14 of 14 files downloaded.  
wandb:   15 of 15 files downloaded.  
wandb:   16 of 16 files downloaded.  
wandb:   17 of 17 files downloaded.  
wandb:   18 of 18 files downloaded.  
wandb:   19 of 19 files downloaded.  
wandb:   20 of 20 files downloaded.  


OK  rows=3000  unique@RTA=294
   α=0.5 random seed=456 (t2cilmtb) ... 

...

OK  rows=3000  unique@RTA=78
     IID tifl   seed= 42 (k9g4zmfx) ... 

wandb:   1 of 1 files downloaded.  
wandb:   2 of 2 files downloaded.  
wandb:   3 of 3 files downloaded.  
wandb:   4 of 4 files downloaded.  
wandb:   5 of 5 files downloaded.  
wandb:   6 of 6 files downloaded.  
wandb:   7 of 7 files downloaded.  
wandb:   8 of 8 files downloaded.  
wandb:   9 of 9 files downloaded.  
wandb:   10 of 10 files downloaded.  
wandb:   11 of 11 files downloaded.  
wandb:   12 of 12 files downloaded.  
wandb:   13 of 13 files downloaded.  
wandb:   14 of 14 files downloaded.  
wandb:   15 of 15 files downloaded.  
wandb:   16 of 16 files downloaded.  
wandb:   17 of 17 files downloaded.  
wandb:   18 of 18 files downloaded.  
wandb:   19 of 19 files downloaded.  
wandb:   20 of 20 files downloaded.  


OK  rows=3000  unique@RTA=244
     IID oort   seed= 42 (39me4hby) ... 

wandb:   1 of 1 files downloaded.  
wandb:   2 of 2 files downloaded.  
wandb:   3 of 3 files downloaded.  
wandb:   4 of 4 files downloaded.  
wandb:   5 of 5 files downloaded.  
wandb:   6 of 6 files downloaded.  
wandb:   7 of 7 files downloaded.  
wandb:   8 of 8 files downloaded.  
wandb:   9 of 9 files downloaded.  
wandb:   10 of 10 files downloaded.  
wandb:   11 of 11 files downloaded.  
wandb:   12 of 12 files downloaded.  
wandb:   13 of 13 files downloaded.  
wandb:   14 of 14 files downloaded.  
wandb:   15 of 15 files downloaded.  
wandb:   16 of 16 files downloaded.  
wandb:   17 of 17 files downloaded.  
wandb:   18 of 18 files downloaded.  
wandb:   19 of 19 files downloaded.  
wandb:   20 of 20 files downloaded.  


OK  rows=3000  unique@RTA=300


In [20]:
# Sanity Check
alpha = 2.0
print(f"Selection-count distribution at {ALPHA_LABELS[alpha]} (pooled across seeds, up to RTA)\n")
print(f"{'Strategy':<8} {'mean':>6} {'std':>6} {'max':>4} {'never':>6}")
print("-" * 36)
for strategy in STRATEGIES:
    pooled = [all_results[alpha][strategy][s]["sel_counts_rta"]
              for s in get_seeds(alpha, strategy)
              if all_results[alpha][strategy][s].get("sel_counts_rta") is not None]
    if not pooled:
        continue
    arr = np.concatenate(pooled)
    print(f"{strategy:<8} {arr.mean():>6.2f} {arr.std():>6.2f} {arr.max():>4d} {int((arr==0).sum()):>6d}")


Selection-count distribution at α=2 (pooled across seeds, up to RTA)

Strategy   mean    std  max  never
------------------------------------
random     2.13   1.42    8     97
fedcs      3.10   6.19   27    672
tifl       2.50   1.94    8    136
oort       2.76   2.27   11      0


In [21]:
all_results[2.0]["oort"][42]['history_table']

,round,node_id,train_loss,duration,compute_time,communication_time,num_samples,selected_by
0,1,9,1.384987,169.453662,166.369210,3.084452,442,"Oort(K=30,eps=0.90,alpha=2.0)"
1,1,7,1.278981,382.896901,356.298397,26.598504,781,"Oort(K=30,eps=0.90,alpha=2.0)"
2,1,8,1.270340,179.130093,175.164874,3.965219,936,"Oort(K=30,eps=0.90,alpha=2.0)"
3,1,1,1.217362,407.517822,353.448602,54.069220,1158,"Oort(K=30,eps=0.90,alpha=2.0)"
4,1,30,0.975251,840.263663,832.820184,7.443479,1552,"Oort(K=30,eps=0.90,alpha=2.0)"
...,...,...,...,...,...,...,...,...
2994,100,277,0.397954,273.717160,272.945537,0.771623,891,"Oort(K=30,eps=0.00,alpha=2.0)"
2995,100,273,0.399644,631.771973,593.786733,37.985239,1758,"Oort(K=30,eps=0.00,alpha=2.0)"
2996,100,283,0.328879,466.578400,457.816513,8.761887,1829,"Oort(K=30,eps=0.00,alpha=2.0)"
2997,100,287,0.357737,278.259915,259.206962,19.052953,1164,"Oort(K=30,eps=0.00,alpha=2.0)"


In [22]:
with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(all_results, f)
print(f"Updated → {OUTPUT_PATH} (added sel_counts_rta, sel_counts_r_max, history_table)")


Updated → results/mnli_all.pkl (added sel_counts_rta, sel_counts_r_max, history_table)


## Statistical Testing: Paired t-test vs Random

For each metric, compute a paired (one-sample) t-test on the per-seed
differences (Random − strategy). Pairing is valid because the same seeds
run across all strategies. IID has one seed, so it is skipped (n < 2).

In [43]:
with open("results/mnli_all.pkl", "rb") as f:
    all_results = pickle.load(f)

print("Loaded keys:", {str(a): list(all_results[a].keys()) for a in all_results})

Loaded keys: {'0.5': ['random', 'fedcs', 'tifl', 'oort'], '2.0': ['random', 'fedcs', 'tifl', 'oort'], 'iid': ['random', 'fedcs', 'tifl', 'oort']}


In [44]:
from scipy import stats  # used only for the t-distribution p-value lookup

# metric key in all_results -> display name
METRICS_FOR_TEST = {
    "rta":            "RTA",
    "tta":            "TTA",        # stored in seconds; units do not affect p
    "mcd": "MCD",
    "best_acc":       "BestAcc",
    "acc_at_r_max":   "Acc@Rmax",
    "jfi_at_rta":     "JFI@RTA",
    "strag_overhead": "StragOH",
}


def paired_diffs(alpha, strategy, key, reference="random"):
    """Per-seed differences (reference - strategy), paired on common seeds."""
    seeds = [
        s for s in get_seeds(alpha, reference)
        if s in all_results[alpha][strategy]
        and all_results[alpha][strategy][s] is not None
        and all_results[alpha][reference][s].get(key) is not None
        and all_results[alpha][strategy][s].get(key) is not None
    ]
    diffs = [
        all_results[alpha][reference][s][key] - all_results[alpha][strategy][s][key]
        for s in seeds
    ]
    return np.array(diffs, dtype=float), seeds


def paired_ttest(alpha, strategy, key, reference="random"):
    """Manual paired t-test. Returns the full arithmetic, not just p."""
    d, seeds = paired_diffs(alpha, strategy, key, reference)
    n = len(d)
    if n < 2:
        return None

    mean_d = d.mean()                 # step 2: mean of differences
    sd_d   = d.std(ddof=1)            # step 3: sample std (divide by n-1)
    se     = sd_d / np.sqrt(n)        # step 4: standard error
    df     = n - 1                    # degrees of freedom

    if se == 0:                       # all differences identical
        t = 0.0 if mean_d == 0 else np.inf
    else:
        t = mean_d / se               # step 5: t-statistic

    p = 2 * stats.t.sf(abs(t), df)    # step 7: two-tailed p-value

    return {"n": n, "diffs": d, "mean_d": mean_d, "sd_d": sd_d,
            "se": se, "df": df, "t": t, "p": p}


In [45]:
def print_ttests(alpha, reference="random"):
    label = ALPHA_LABELS[alpha]
    print(f"\nPaired t-test vs {reference} — alpha = {label}")
    print(f"{'Strategy':<8} {'Metric':<9} {'n':>2} {'mean_d':>11} "
          f"{'sd_d':>9} {'t':>8} {'p':>8}  sig")
    print("-" * 66)
    for strategy in STRATEGIES:
        if strategy == reference:
            continue
        for key, name in METRICS_FOR_TEST.items():
            r = paired_ttest(alpha, strategy, key, reference)
            if r is None:
                print(f"{strategy:<8} {name:<9}  (n<2, skipped)")
                continue
            p = r["p"]
            sig = "**" if p < 0.01 else "*" if p < 0.05 else "." if p < 0.10 else "ns"
            t_s = "inf" if np.isinf(r["t"]) else f"{r['t']:.2f}"
            print(f"{strategy:<8} {name:<9} {r['n']:>2} {r['mean_d']:>+11.4f} "
                  f"{r['sd_d']:>9.4f} {t_s:>8} {p:>8.4f}  {sig}")
        print("-" * 66)


print_ttests(2.0)
print_ttests(0.5)   # FedCS RTA/TTA auto-skip (None); IID would skip on n<2


Paired t-test vs random — alpha = α=2
Strategy Metric     n      mean_d      sd_d        t        p  sig
------------------------------------------------------------------
fedcs    RTA        3     -9.6667   10.9697    -1.53   0.2665  ns
fedcs    TTA        3 +17684.3773 4876.2144     6.28   0.0244  *
fedcs    MCD        3   +286.2659   15.9268    31.13   0.0010  **
fedcs    BestAcc    3     +0.0216    0.0078     4.81   0.0406  *
fedcs    Acc@Rmax   3     +0.0217    0.0053     7.14   0.0191  *
fedcs    JFI@RTA    3     +0.4755    0.0293    28.09   0.0013  **
fedcs    StragOH    3     +1.1650    0.1677    12.03   0.0068  **
------------------------------------------------------------------
tifl     RTA        3     -3.6667    1.5275    -4.16   0.0533  .
tifl     TTA        3  +9224.3086 4009.8787     3.98   0.0576  .
tifl     MCD        3    -20.1502   87.4535    -0.40   0.7284  ns
tifl     BestAcc    3     +0.0082    0.0031     4.57   0.0447  *
tifl     Acc@Rmax   3     +0.0077    0.0